# Monte Carlo Option Pricing Simulation

This notebook runs **NumPy-vectorized** Monte Carlo simulations on the driver.
No Spark UDFs, no broadcast variables — pure vectorized computation.

| Module | Responsibility |
| --- | --- |
| `config/settings.py` | Table names, export paths, logging |
| `transforms/simulation.py` | 6 vectorized simulation methods (NumPy) |
| `utils/simulation_helpers.py` | Data loading, distribution fitting, risk-free rate fetch, run orchestration, write |

## Simulation Methods

| Method | Description |
| --- | --- |
| `historical` | Random sampling from historical log returns (with replacement) |
| `window` | Consecutive T-day window from random start (wrapping) |
| `window_10d` | Window with 10-day step between samples |
| `window_20d` | Window with 20-day step between samples |
| `student_t` | (1-alt_weight) historical + alt_weight Student-t draws |
| `black_scholes` | GBM under risk-neutral measure (benchmark) — uses r and annualized vol, discounted to PV |

**Parameters (widgets):** ticker, strike price (K), simulation period (T days), number of runs

**Market data:** Risk-free rate fetched live from ^IRX (13-week T-bill), volatility computed from historical log returns (annualized)

**Input:** `default.yfinance_historical_data`  
**Output:** `default.simulation_results`

In [0]:
%pip install scipy yfinance --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os

src_dir = "/Workspace/Users/spyros.louvis@ctpsandbox.com/pub-python-repo/databricks/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from config.settings import FULL_TABLE_NAME, FULL_RESULTS_TABLE_NAME, get_export_path, get_logger
from utils.simulation_helpers import (
    load_historical_data, fit_distributions, run_simulations,
    results_to_spark, write_results, get_risk_free_rate,
)

logger = get_logger("monte_carlo_simulation")
logger.info("Modules loaded successfully.")

In [0]:
# ---------------------------------------------------------------------------
# Widgets (parameterized for Job runs or interactive use)
# ---------------------------------------------------------------------------
dbutils.widgets.text("ticker", "^GSPC", "Ticker Symbol")
dbutils.widgets.text("target_value", "5700", "Strike Price K")
dbutils.widgets.text("period", "10", "Simulation Period T (days)")
dbutils.widgets.text("num_simulations", "1000", "Number of Simulations")

ticker = dbutils.widgets.get("ticker")
K = float(dbutils.widgets.get("target_value"))
T = int(dbutils.widgets.get("period"))
num_runs = int(dbutils.widgets.get("num_simulations"))
alt_weight = 0.1

logger.info(f"Parameters: ticker={ticker}, K={K}, T={T}, runs={num_runs}, alt_weight={alt_weight}")

In [0]:
# Load historical log returns and fit distributions
log_returns, S0 = load_historical_data(spark, FULL_TABLE_NAME, ticker)
dist_params = fit_distributions(log_returns)

# Fetch risk-free rate for Black-Scholes benchmark
r = get_risk_free_rate()

logger.info(f"Data loaded. S0={S0:.2f}, vol={dist_params['vol']:.4f}, r={r:.4f}")

In [0]:
# Run all simulation methods (NumPy vectorized on driver)
results_pdf = run_simulations(
    log_returns, S0, K, T, num_runs,
    dist_params=dist_params,
    alt_weight=alt_weight,
    r=r,
)
display(spark.createDataFrame(results_pdf))

In [0]:
import numpy as np
from scipy.stats import norm

# ---------------------------------------------------------------------------
# Black-Scholes Analytical Formula (European options)
# ---------------------------------------------------------------------------
vol = dist_params["vol"]  # annualized volatility
T_years = T / 252.0       # convert trading days to years

d1 = (np.log(S0 / K) + (r + 0.5 * vol**2) * T_years) / (vol * np.sqrt(T_years))
d2 = d1 - vol * np.sqrt(T_years)

# Analytical prices
bs_call = S0 * norm.cdf(d1) - K * np.exp(-r * T_years) * norm.cdf(d2)
bs_put = K * np.exp(-r * T_years) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

# MC result from simulation
mc_row = results_pdf[results_pdf["method"] == "black_scholes"].iloc[0]
mc_call = mc_row["CallPrice"]
mc_put = mc_row["PutPrice"]

# Standard error estimate (from MC variance)
log_sums_bs = (r - 0.5 * vol**2) * (1/252) * T + vol * np.sqrt(1/252) * np.random.normal(size=num_runs)
log_sums_bs = np.clip(log_sums_bs, -10.0, 10.0)
final_prices_bs = S0 * np.exp(log_sums_bs)
call_payoffs_bs = np.maximum(final_prices_bs - K, 0.0) * np.exp(-r * T_years)
se_call = float(np.std(call_payoffs_bs) / np.sqrt(num_runs))
put_payoffs_bs = np.maximum(K - final_prices_bs, 0.0) * np.exp(-r * T_years)
se_put = float(np.std(put_payoffs_bs) / np.sqrt(num_runs))

# Comparison table
print(f"{'='*65}")
print(f"BLACK-SCHOLES: ANALYTICAL vs MONTE CARLO ({num_runs:,} paths)")
print(f"{'='*65}")
print(f"  S0 = ${S0:,.2f}  |  K = ${K:,.2f}  |  T = {T} days ({T_years:.4f} yrs)")
print(f"  r  = {r:.4f}    |  vol = {vol:.4f} (annualized)")
print(f"  d1 = {d1:.6f}  |  d2 = {d2:.6f}")
print(f"{'─'*65}")
print(f"  {'':20s} {'Analytical':>12s} {'MC Estimate':>12s} {'Diff':>8s} {'SE':>8s}")
print(f"  {'Call Price':20s} ${bs_call:>10.4f}  ${mc_call:>10.4f}  {mc_call - bs_call:>+7.2f}  ±{se_call:.2f}")
print(f"  {'Put Price':20s} ${bs_put:>10.4f}  ${mc_put:>10.4f}  {mc_put - bs_put:>+7.2f}  ±{se_put:.2f}")
print(f"{'─'*65}")
print(f"  Call MC error: {abs(mc_call - bs_call) / bs_call * 100:.2f}%  "
      f"({'within' if abs(mc_call - bs_call) < 2 * se_call else 'outside'} 2 SE)")
print(f"  Put  MC error: {abs(mc_put - bs_put) / bs_put * 100:.2f}%  "
      f"({'within' if abs(mc_put - bs_put) < 2 * se_put else 'outside'} 2 SE)")

# ---------------------------------------------------------------------------
# Put-Call Parity Check: C - P = S0 - K*exp(-rT)
# ---------------------------------------------------------------------------
parity_theoretical = S0 - K * np.exp(-r * T_years)
parity_analytical = bs_call - bs_put
parity_mc = mc_call - mc_put

print(f"\n{'='*65}")
print(f"PUT-CALL PARITY: C - P = S0 - K*exp(-rT)")
print(f"{'='*65}")
print(f"  Theoretical:  S0 - K*exp(-rT) = ${parity_theoretical:>10.4f}")
print(f"  Analytical:   C  - P          = ${parity_analytical:>10.4f}  "
      f"(err: ${abs(parity_analytical - parity_theoretical):.6f})")
print(f"  Monte Carlo:  C  - P          = ${parity_mc:>10.4f}  "
      f"(err: ${abs(parity_mc - parity_theoretical):.2f})")
print(f"{'─'*65}")
parity_err_pct = abs(parity_mc - parity_theoretical) / abs(parity_theoretical) * 100
parity_ok = parity_err_pct < 5.0
print(f"  MC parity deviation: {parity_err_pct:.2f}%  "
      f"{'\u2705 PASS' if parity_ok else '\u26a0\ufe0f CHECK'} (threshold: 5%)")
print(f"{'='*65}")

In [0]:
# Convert to Spark DataFrame with metadata and write
results_df = results_to_spark(spark, results_pdf, ticker, S0)
export_path = get_export_path(ticker)
write_results(results_df, FULL_RESULTS_TABLE_NAME, ticker, export_path)
logger.info(f"[DONE] Simulation complete for ticker={ticker}, K={K}, T={T}, runs={num_runs}")